# Deploying Granite 4.0 h tiny models in Amazon SageMaker

## Introduction to Granite 4.0 h tiny Models

IBM Granite 4.0 h tiny represents the next evolution in the Granite family of models.  
These models are optimized for:
- Low latency
- Cost-efficient inference
- Instruction-following tasks
- Tool use, summarization, classification, entity extraction, and RAG.

The Granite 4.0 series has been trained with high-quality data using an optimized training pipeline to ensure strong performance relative to its compact size. Like previous releases, these models are available under Apache 2.0 license on SageMaker JumpStart, AWS Marketplace, and [Hugging Face](https://huggingface.co/ibm-granite).

---

## Pre-requisites

- Use this notebook from **Amazon SageMaker Notebook Instance** or **SageMaker Studio**.
- Ensure your execution IAM role has **AmazonSageMakerFullAccess**.
- Subscribe to the model package on AWS Marketplace.

---

## Contents
1. Subscribe to and select the Granite 4.0 h tiny model
2. Create an endpoint
3. Run real-time inference
4. Clean up resources

---

### 1. Subscribe to the model package

1. Go to [IBM Granite 4.0 h tiny on AWS Marketplace](https://aws.amazon.com/marketplace).
2. Click **Continue to subscribe**.
3. Accept the offer, choose your region, and copy the **Product ARN**.
4. Paste the ARN into the `model_package_map` below for your region.

### 1. Select the model package

Confirm that you received this notebook from model catalog on SageMaker AWS Management Console.

In [ ]:
model_package_map = {
    "us-east-1": "",
    "us-east-2": "",
    "us-west-1": "",
    "us-west-2": "",
    "ca-central-1": "",
    "eu-central-1": "",
    "eu-west-1": "",
    "eu-west-2": "",
    "eu-west-3": "",
    "eu-north-1": "",
    "ap-southeast-1": "",
    "ap-southeast-2": "",
    "ap-northeast-2": "",
    "ap-northeast-1": "",
    "ap-south-1": "",
    "sa-east-1": ""
}

In [ ]:
!pip install --upgrade pip
!pip install -U sagemaker -q

In [ ]:
import json
import pprint
from datetime import datetime

import boto3
import sagemaker
from sagemaker import ModelPackage, get_execution_role

In [ ]:
sagemaker_session = sagemaker.Session()

try:
    execution_role_arn = sagemaker.get_execution_role()
except ValueError:
    execution_role_arn = None

if execution_role_arn == None:
    execution_role_arn = input("Enter your execution role ARN: ")

region = sagemaker_session.boto_region_name
runtime_sm_client = boto3.client("runtime.sagemaker")

print ("execution_role_arn: ", execution_role_arn)
print ("region: ", region)

In [ ]:
if region not in model_package_map.keys():
    raise "UNSUPPORTED REGION"

model_package_arn = model_package_map[region]

print ("model_package_arn: ", model_package_arn)

## Create an endpoint and perform real-time inference

In this example, we’re deploying IBM Granite-4.0-h-tiny on an Amazon SageMaker real-time endpoint hosted on a GPU instance.
If you need general information on real-time inference with Amazon SageMaker, please refer to the SageMaker documentation.

The Granite 4.0 h tiny model is part of IBM’s fourth-generation Granite family. It is a compact, instruction-tuned language model designed to deliver high performance with minimal compute requirements. This makes it ideal for cost-efficient, low-latency inference in enterprise scenarios such as summarization, classification, tool use, and RAG pipelines.

For flexibility, you can pick from different instance configurations depending on your workload and availability.
In this example, we use an ml.g6e.2xlarge instance, which provides an NVIDIA L40S GPU. This instance type offers an excellent balance between cost and performance, making it well suited for lightweight yet production-grade deployments.

The IBM Granite-4.0-h-tiny model supports:
	•	Instruction-following tasks
	•	Low-latency generation
	•	Long-context input handling
	•	Efficient scaling for production workloads

### 2. Define the endpoint configuration

In [ ]:
model_name = "granite-4-0-h-tiny"
inference_instance_type = "ml.g6e.2xlarge"
model_download_timeout = 3600
health_check_timeout = 900
instance_count = 1

### 3. Create the endpoint

In [ ]:
# create a deployable model from the model package.
model = ModelPackage(
    role=execution_role_arn, model_package_arn=model_package_arn, sagemaker_session=sagemaker_session
)

# create a unique endpoint name
timestamp = "{:%Y-%m-%d-%H-%M-%S}".format(datetime.now())
endpoint_name = f"{model_name}-{timestamp}"
print(f"Deploying endpoint {endpoint_name}")

In [ ]:
# deploy the model
deployed_model = model.deploy(
    initial_instance_count=instance_count,
    instance_type=inference_instance_type,
    endpoint_name=endpoint_name
)


If you have already deployed your model, you can also access it via your chosen endpoint_name and sagemaker_session:

In [ ]:
deployed_model = sagemaker.Predictor(
    endpoint_name=endpoint_name,
    sagemaker_session=sagemaker_session,
)

SageMaker will now create our endpoint and deploy the model to it. This can takes a 10-15 minutes. Once the endpoint is in service, you will be able to perform real-time inference.

## Run inference with the model

Now that we have the Granite 4.0 h tiny instruct model loaded and deployed to a SageMaker endpoint, we can start testing. We use the predict method from the predictor to run inference on our endpoint. We can inference with different parameters to impact the generation. Parameters can be defined as in the parameters attribute of the payload.

### 4. This is an example model request! I think LLMs are cool. 

In this example, we want to test sample prompt.

In [ ]:
prompt_1 = """This is an example model request! I think LLMs are cool."""

In [ ]:
# hyperparameters for llm
payload_1 = {
    "inputs": prompt_1,
    "parameters": {
        "details": True,
        "max_tokens": 512
    }
}

# send request to endpoint
response_1 = deployed_model.predict(
    data=json.dumps(payload_1),
    initial_args={"Accept": "application/json", "ContentType": "application/json"},
).decode("utf-8")

response_list = json.loads(response_1)[0]
print(response_list["generated_text"])


## Clean-up

Please don't forget to run the cells below to delete all resources and avoid unecessary charges.

### 7. Delete the endpoint

In [ ]:
model.sagemaker_session.delete_endpoint(endpoint_name)
model.sagemaker_session.delete_endpoint_config(endpoint_name)

### 8. Delete the model

In [ ]:
model.delete_model()

Thank you for trying out IBM Granite 4.0 h tiny Model on SageMaker. We have only scratched the surface of what you can do with this model.

Welcome to your IBM Granite Model support experience! You can view, start, or contribute to community discussions (sign in to contribute). View supplemental resources and  [sign](https://www.ibm.com/mysupport/s/?language=en_US) in to open a new case.

## Would you like to provide feedback?

Please let us know your comments about our family of Granite 4.0 h tiny instruct models by visiting our collection. Select the repository of the model you would like to provide feedback about. Then, go to Community tab, and click on New discussion. Alternatively, you can also post any questions/comments on our github discussions page.